# Movie Lens MLOps with TFX

The goal of a recommender system for Movie Lens is to return personalized movie recommendations to a user for movies they haven't seen.

This notebook trains a bi-encoder using contrastive listwise learning in order to have Query and Candidate models which can, given user or movie inputs, return embeddings that can be used to find approximate nearest neighbors in terms of users or movies.

The metric used to select the best fitting nodel is NDCG@K with K=20.
Aside from a requirement of NDCG@K to be at least 3 times better than performance for random selection as current baseline, a production model would also need to satisfy business requirements.
See the section below called "Linking to business value".

The MLOPs pipeline used is TFX.

The Movie Lens data is extracted, analyzed, processed and stored and registered with an MLMetadata registry.  TwoTowerDNN contrastive listwise models are trained and tuned
with an in-batch softmax loss.  The best models are evaluated using NDCG@20, validated, and deployed to storage, MLMD registry, and/or serving endpoints. 

The deployed models should be monitored for model degradation and anomalies.  The live 
incoming data should also be monitored for shifts in distributions.
The model validation should involve testing, including A/B testing if possible and manual confirmation that the
model aligns with business objectives.

All of these steps are called machine learning operations (MLOps) and are automated when possible.
There are iterative and continuous loops in segments of a total MLOps pipeline or pipelines.
The source code for this notebook and pipelines are [at github, as recommender_systems](https://github.com/nking/recommender_systems)

The associated notebook summarizing recommender models, and using the results of this one to continue with a Retrieval Model and Re-Ranker model is [Recommender Systems ](https://www.kaggle.com/code/nicholeasuniquename/recommender-systems/).

## Kaggle notebook
1. Create a virtual environment for TFX compatability.
2. Download the pipeline code from the [public repository](https://github.com/nking/recommender_systems).
3. EDA on the raw data, joined data, and transformed data
4. Define Data and Concept Drifts
5. Run the data pre-processing pipeline and look at results.
6. Run the Baseline and Production model pipeline and look at results.
7. Explore the model evaluation
8. Model Analysis
9. Link to Business Value
10. Monitor the model

## Use CPU or GPU?
**You should use the CPU for this notebook.**
  * but if I were to update the code to latest TFX pipeline which is now available and check compatibility for kaggle's ubuntu OS C library with that version, and if I were to modify the _get_stratgey() method to use tf.distribute.OneDeviceStrategy("/gpu:0")  then one could use the T4x2 GPU selection here instead.  (Note I chose to design for CPU originally due to TFX 2.16 avail at time and the lower runtime complexity of training previously).

Here are the arguments for using the **T4x2 GPU(s)** instead if designing the project starting now (making those changes above).

While single-query inference (`call`) is lightweight, full-batch contrastive training (`train_step` at $B = 2048$) with Wang & Isola hypersphere uniformity, Top-$K$ Intra-List Diversity (`ILD@20`), and 8 in-batch ranking metrics is **$16\times$ to $25\times$ faster on a T4 GPU than on the Kaggle CPU**.

### Hardware Specifications & Roofline Ridge Points
- **Arithmetic Intensity ($I$)** $= \frac{\text{FLOPs}}{\text{Memory Traffic (Bytes Loaded + Stored)}}$
- **Hardware Ridge Point ($I_{\text{ridge}}$)** $= \frac{\text{Peak Hardware FLOPs/sec}}{\text{Peak Memory Bandwidth Bytes/sec}}$ (see [Roofline Analysis](https://docs.nersc.gov/tools/performance/roofline/))
  * **NVIDIA T4 GPU (FP32):** $8{,}100\text{ GFLOPS (8.1 TFLOPS)} \;/\; 320\text{ GB/s} = \mathbf{25.3\text{ FLOPs/Byte}}$
  * **Kaggle CPU (4-vCPU Xeon):** $100\text{ GFLOPS} \;/\; 20\text{ GB/s} = \mathbf{5.0\text{ FLOPs/Byte}}$

### "Memory-Bound on GPU" Still Beats "Compute-Bound on CPU"
In Roofline analysis, achievable performance is:
$$\text{Attainable Throughput (GFLOPS)} = \min\!\big(\text{Peak GFLOPS},\; I \times \text{Bandwidth}\big)$$
When an operation has Arithmetic Intensity $I < 25.3\text{ FLOPs/Byte}$, it is memory-bound on the T4 GPU—meaning its speed is dictated by **memory bandwidth**. Because the T4 GPU has **$16\times$ higher memory bandwidth** than the Kaggle CPU ($320\text{ GB/s}$ vs. $20\text{ GB/s}$) and **$81\times$ higher compute** ($8{,}100\text{ GFLOPS}$ vs. $100\text{ GFLOPS}$), the T4 GPU outperforms the CPU at **every** Arithmetic Intensity $I > 0$:
* At $I = 7.9\text{ FLOPs/Byte}$ (single $Q C^T$ matmul):
  * **T4 GPU Throughput:** $\min(8{,}100,\; 7.88 \times 320) = \mathbf{2{,}521\text{ GFLOPS}}$ *(memory-bound, running at 31% of peak compute)*
  * **Kaggle CPU Throughput:** $\min(100,\; 7.88 \times 20) = \mathbf{100\text{ GFLOPS}}$ *(compute-bound, capped at 100% of CPU ceiling)*
  * **Result:** The T4 GPU is **$25.2\times$ faster** than the CPU.

### Algorithmic Intensity of `TwoTowerDNN.train_step` ($B = 2048, D = 16, K_{\text{ILD}} = 20$)
1. **All-to-All Embedding Matmuls (7 passes of $[B, D] \times [D, B]$ across Fwd + Bwd):**
   * Forward: `raw_logits` ($U M^T$), user uniformity `u_sim` ($U U^T$), movie uniformity `m_sim` ($M M^T$) $= 3 \times 2 B^2 D$
   * Backward (`GradientTape`): $\nabla_U, \nabla_M$ for `raw_logits` (2 matmuls), `u_sim` (1 matmul), `m_sim` (1 matmul) $= 4 \times 2 B^2 D$
   * **FLOPs:** $7 \times 2 \times 2048^2 \times 16 = \mathbf{939{,}524{,}096\text{ FLOPs}}$ ($939.5\text{ MFLOPs}$)
   * **Bytes:** $7 \times (2 B D + B^2) \times 4 = \mathbf{119{,}275{,}520\text{ Bytes}}$ ($119.3\text{ MB}$)
   * **Arithmetic Intensity:** $\mathbf{7.88\text{ FLOPs/Byte}}$ (approaching $D/2 = 8.0$ as $B \to \infty$)

2. **In-Batch Top-$K$ Intra-List Diversity (`ILD@20`):**
   * Partial heap-select `top_k(k=20)` over $[B, B]$ ($B^2 \lceil\log_2 20\rceil = 21.0\text{ MOPS}$) + gather $[B, 20, D]$ + batched slate matmul $[B, 20, D] \times [B, D, 20] \to [B, 20, 20]$ (Fwd + Bwd $= 4 B K^2 D = 52.4\text{ MFLOPs}$) + distance masking ($4.9\text{ MFLOPs}$)
   * **FLOPs:** $\mathbf{78{,}315{,}520\text{ FLOPs}}$ ($78.3\text{ MFLOPs}$)
   * **Bytes:** $(B^2 + 2 B K + 3 B K D + 4 B K^2) \times 4 = \mathbf{38{,}076{,}416\text{ Bytes}}$ ($38.1\text{ MB}$)
   * **Arithmetic Intensity:** $\mathbf{2.06\text{ FLOPs/Byte}}$

3. **Dense $[B, B]$ Elementwise & Reduction Kernels (Uniformity $\exp$, Softmax, Masking & 8 Ranking Metrics):**
   * Gaussian potential $\exp(-4 + 4\cdot\text{sim})$ on `u_sim` and `m_sim` + Fwd/Bwd softmax on `masked_logits` and `masked_raw_logits` (column entropy) + $O(B^2)$ false-negative masking + rank comparisons (`>= pos_scores`) across `mrr_100`, `ndcg_100`, `head`, `torso`, `tail`, `recall_100`, `precision_100`, and `hit_rate`
   * **FLOPs:** $\sim 74 \times B^2 = \mathbf{310{,}378{,}496\text{ FLOPs}}$ ($310.4\text{ MFLOPs}$)
   * **Bytes:** $\sim 16 \times B^2 \times 4 = \mathbf{268{,}435{,}456\text{ Bytes}}$ ($268.4\text{ MB}$)
   * **Arithmetic Intensity:** $\mathbf{1.16\text{ FLOPs/Byte}}$

4. **Query & Candidate Tower MLPs (`[B, 44] -> [B, 32] -> [B, 16]` Fwd + Bwd):**
   * **FLOPs:** $\mathbf{36{,}175{,}872\text{ FLOPs}}$ ($36.2\text{ MFLOPs}$)

### Total `train_step` Roofline Comparison ($B = 2048$)
* **Total FLOPs per step:** $\mathbf{1.36\text{ GFLOPs}}$ ($1{,}364{,}393{,}984\text{ FLOPs}$)
* **Total Memory Traffic per step:** $\mathbf{425.8\text{ MB}}$ ($425{,}787{,}392\text{ Bytes}$)
* **Composite Arithmetic Intensity:** $1{,}364.4\text{ MFLOPs} \;/\; 425.8\text{ MB} = \mathbf{3.20\text{ FLOPs/Byte}}$
* **Attainable Throughput & Step Time:**
  * **Kaggle CPU:** $\min(100,\; 3.20 \times 20) = \mathbf{64.0\text{ GFLOPS}} \implies \sim\mathbf{180\text{–}220\text{ ms/step}}$ ($\sim 45\text{s/epoch}$, or $\sim 10\text{ hours}$ for 40 HPO trials)
  * **NVIDIA T4 GPU:** $\min(8{,}100,\; 3.20 \times 320) = \mathbf{1{,}024\text{ GFLOPS}} \implies \sim\mathbf{8\text{–}15\text{ ms/step}}$ ($\sim 2\text{–}3\text{s/epoch}$, completing all 40 HPO trials in **$< 45\text{ minutes}$**)

## 1. Creating the TFX compatible virtual environment

A conda virtual env is used for version control of python and the python packages.

To run the TFX pipeline on the current kaggle host which has Ubuntu 22.04 installed as of Aug 2026,
a virtual environment for python 3.10 is needed due to ml-metadata and protobuf version compatibility constraints within dependencies.

Once conda is installed and a virtual environment is created for the fixed version of python, the virtual environment can be activated by activating conda and then activating the virtual environment.

A bash shell in the notebook that is invoked from the magic command %%bash is a bash session for the extent of that specific cell.
For each new session invoked by the cell %%bash, the 2 activation commands need to be invoked before using the virtual environment.

Aside from running scripts in the magic bash shell cells, we can also run scripts using the python subprocess library as long as we prepend commands with the 2 conda activation statements.

We have 2 ways to run commands within the virtual environment.

The notebook itself is still using the kaggle docker image environment without the newly built virtual environment.
Even if we install and use ipykernel to register a kernel for the new virtual environment, I don't see a way to open the notebook to use the new kernel.  (In the Kaggle window, we have Session options, persistence option to persist files and variables, so it might be possible to restart the notebook with kernel selected as long as the kernel has Kaggle specific notebook support...)

In summary, the notebook as is can be used for intermediate steps of EDA where the EDA uses libraries that don't require an earlier version of python.  For MLOps steps that need an earlier version of python, the virtual environment is available.


**Alternative (commercial) notebook platforms** with selectable notebook kernels:
* Google Vertex AI Workbench instances
    * [Add a conda environment](https://cloud.google.com/vertex-ai/docs/workbench/instances/add-environment)
    * (see tutorials in [QwikLabs](https://www.skills.google/) and [Coursera](coursera.org))
  
    Note that, Vertex AL also has "low-code" and "no-code" modeling options with [BigQueryML](https://docs.cloud.google.com/bigquery/docs/bqml-introduction) and [AutoML](https://docs.cloud.google.com/vertex-ai/docs/training/automl-training-overview).
* [Colab enterprise](https://docs.cloud.google.com/colab/docs/introduction)
* [Amazon SageMaker](https://aws.amazon.com/blogs/machine-learning/four-approaches-to-manage-python-packages-in-amazon-sagemaker-studio-notebooks/)
* [Azure Machine Learning](https://learn.microsoft.com/en-us/azure/machine-learning/concept-environments?view=azureml-api-2)

In [ ]:
!pwd
!echo $HOME

%%bash: Executes the entire cell as a shell script. 

In [ ]:
%%bash
t0=$(date +%s%N)
mkdir -p ~/miniconda3
wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O ~/miniconda3/miniconda.sh
#install conda and activate to /usr/local
bash ~/miniconda3/miniconda.sh -b -u -p /usr/local
rm ~/miniconda3/miniconda.sh
conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r

. /usr/local/bin/activate
echo "**$SHELL**"
echo "**$BASH**"
conda init --all

. /root/.bashrc
conda create -q --name my_tfx_env python=3.10 -y
conda activate my_tfx_env
python --version

t1=$(date +%s%N)
t2=$(echo "scale=9;($t1-$t0) / 1000000000" | bc)
echo $t2 seconds
date

In [ ]:
%%bash
. /usr/local/bin/activate
conda activate my_tfx_env
conda install pip -y -q
conda install ipykernel -y -q

## 2. Download the pipeline code from my [public repository](https://github.com/nking/recommender_systems).

The repository contains the MLOps pipelines built with custom TFX components and scripts defining the data preprocessing, hyper-parameter tuning, model building, data and model evaluations, and model monitoring.

The pipelines are built to use distributed data processing and distributed model training.

The model being tuned and trained is a multi-tower model, 2 towers specifically, and is also known as a bi-encoder.



To activate the conda environment, you need to source from conda's activate (which I installed in /usr/local/bin above), then activate the conda virtual environment.

This has to be done for each magic shell cell.

In [ ]:
%%bash
t0=$(date +%s%N)
. /usr/local/bin/activate
conda activate my_tfx_env
python --version

echo "this cell takes 7-ish minutes to run"

#consider conda install ipykernel
conda install pip -y -q
conda install ipykernel -y -q
echo "install git"
conda install git -y -q

#conda config --add channels conda-forge
#conda config --set channel_priority strict
#conda install python-snappy
# or:
#conda install anaconda::python-snappy


DIRECTORY="/kaggle/working/recommender_systems"

if [ -d "$DIRECTORY" ]; then
  echo "Directory '$DIRECTORY' exists."
else
  echo "cloning the repository"
  git clone --depth 1 https://github.com/nking/recommender_systems.git
fi

pip list
conda env list

python --version
java --version

t1=$(date +%s%N)
t2=$(echo "scale=9;($t1-$t0) / 60000000000" | bc)
echo "$t2 minutes"
date
#about 6-7 minutes for this cell.

### Install the Movie Lens TFX pipeline

In [ ]:
%%bash
. /usr/local/bin/activate
conda activate my_tfx_env

date

cd recommender_systems

# HERE, if running on GPUs, change -kaggle.py to -kaggle-gpu.py
mv setup-kaggle.py setup.py
mv pyproject-kaggle.toml pyproject.toml

echo "installing the code as a pip package"
pip3 install .
#pip install --editable .

pip show movie_lens_tfx

## 3. EDA on the raw data, joined data, and transformed data

The data are from [GroupLens](https://files.grouplens.org/datasets/movielens/)
"1m" dataset.

The datasets are split per user by their timestamps as 80:10:10 perent into train, val, and test.  This helped have beeter overlap for downstream interaction in the cross-encoder.

### 3.a. Joined data from the MovieLensExampleGen component
A custom TFX component was created to left inner join the ratings with the user and movie data using Apache Beam PTransforms. The custom python function version of the component is [here](https://github.com/nking/recommender_systems/tree/main/src/main/python/movie_lens_tfx/ingest_component) and the fully custom component is [here](https://github.com/nking/recommender_systems/tree/main/src/main/python/movie_lens_tfx/ingest_pyfunc_component).  They both use [PTransforms here](https://github.com/nking/recommender_systems/blob/main/src/main/python/movie_lens_tfx/utils/ingest_movie_lens_beam.py). Only the custom python function is used in the pipeline, but it was instructive to create both component versions.

That ingest component was then replaced for the TwoTowerDNN pipeline by a similar ingest component, but this time constructed to accept data already split into train, val, and test datasets.  That component is [here](https://github.com/nking/recommender_systems/tree/main/src/main/python/movie_lens_tfx/ingest_already_split_pyfunc_component).

### 3.b. Exploration

Plots were made using [Tensorboard](https://www.tensorflow.org/tensorboard), [polars](https://pola.rs/), [plotly.express](https://plotly.com/python/plotly-express/), [TFDV](https://www.tensorflow.org/tfx/data_validation/get_started), [Apache Spark MLLib FPGrowth](https://spark.apache.org/docs/latest/mllib-frequent-pattern-mining.html), and [pandas](https://pandas.pydata.org/) (displayed from files generated on a computer with notebook having kernel of the virtual environment).  

see scripts in src/main/python/eda/* and the notebook recommender-systems-with-tfx-pipelines.ipynb

#### Data Shift in the train, val, test splits.
(these are fom the script in the retrieval project called post_training_analysis.py

* "HIGH MOVIE TIER DRIFT WARNING: Significant item popularity shift between Train and Test splits (Train-Test EMD: 0.0679). Evaluation may be biased."
* "USER TIER STABILITY EXCELLENT: Activity profile distribution remains consistent across splits (Train-Test EMD: 0.0113)."
* "EXPECTED INTERACTION POPULARITY BIAS: Catalog vs. Train EMD is 0.4699. Training interactions heavily concentrate on high-tier items relative to catalog availability."
* "P_X MOVIE TIER HIGH DRIFT WARNING: Significant item popularity shift between Train and Test splits (Train-Test EMD: 0.0932)."
* "P_Y|X MOVIE TIER STABILITY EXCELLENT: Activity profile distribution remains consistent across splits (Train-Test EMD: 0.0049)."
* "P_Y MOVIE TIER STABILITY EXCELLENT: Activity profile distribution remains consistent across splits (Train-Test EMD: 0.0000)."
* "P_X|Y MOVIE TIER HIGH DRIFT WARNING: Significant item popularity shift between Train and Test splits (Train-Test EMD: 0.0679).",
* ==> "Covariate shift between Train and Test splits."

NOTE one solution to data shift used commercial large applications is to perform data splits by rolling time windows of small duration but large amount of data.  The train split is additionally enriched by random sampling of user history of past 1+ years to capture seasonality.  The validation split is strictly later in timestamps than train and the test dataset is stricly later in timestamp than validation. The model is continuously trained with the new splits.

#### Roughly scraped TFDV snapshots of pre-processing histograms:
<img src="https://raw.githubusercontent.com/nking/recommender_systems/refs/heads/main/src/test/resources/pre_processed_tfdv_hist.png">

#### Using Polars and plotly.express and distance correlation:
* You can see a clear correlation between rating and movie_id, then less
so between rating and age and rating and occupation.

<img src="https://raw.githubusercontent.com/nking/recommender_systems/refs/heads/main/src/test/resources/train_dist_corr_heatmap.png">

* You can see many co-occurrences between movie genres.

<img src="https://raw.githubusercontent.com/nking/recommender_systems/refs/heads/main/src/test/resources/train_genre_cooccurence_rating_5_heatmap.png">

* You can see that Drama, then Comedy, then Action are the most popular movie genres.

<img src="https://raw.githubusercontent.com/nking/recommender_systems/refs/heads/main/src/test/resources/train_genres_rating_5.png">

#### using Pandas, Pyspark MLLIB FPGrowth

From the eda_transformed_pyspark_mllib.py script, a market basket analysis with movie_ids.

* You can see that many association rules can be derived from the data.
<img src="https://raw.githubusercontent.com/nking/recommender_systems/refs/heads/main/src/test/resources/train_movies_assoc_rules_rating_5.png">

* and that there are many frequent itemsets.
<img src="https://raw.githubusercontent.com/nking/recommender_systems/refs/heads/main/src/test/resources/train_movies_itemsets_rating_5.png">

<img src="https://raw.githubusercontent.com/nking/recommender_systems/refs/heads/main/src/test/resources/train_movies_itemsets_rating_5_2.png">

#### Ratings movie frequency distribution
The frequency distribution of the training dataset of positive ratings was partitioned into the top 20%, between the top 20% and the bottom 20% and the bottom 20% to add tiers of head, torso and tail respectively to the movie catalog.  Movies not present in the train distributiona are assigned to the tail tier.

A quick look at the counts of the distribution in the training and validation datasets in order to determine whether learning the tail of the distribution is feasible.

In the following, intersect refers to intersection of the train and val datasets by the specific characteristic such as movie_id or user_id. tier=0 is head, tier=1 is torso, tier=2 is tail of movie ratings frequency distribution.

* number of unique movies = 3883
* ratings in train dataset, filtered to intersection of train and val by movies=458445
* ratings in val  ... =51571
* ratings in train intersect by movies for tier=2=4330
* ratings in val  ... =1055
* unique users intersection of train and val after unique movies intersection: 5135
* ratings in train intersect by movies then users=454245
* ratings in val ... =51571
* ratings in train intersect by movies then users for tier=2=4291
* ratings in val ... =1055
* unique movies in intersection of train and val by movies then users for tier=2: 446
* unique users ... =2: 414

The later 4 counts show that there are enough ratings of tier=2 movies
to learn the niche (tail) recommendations and hence learn recommendations for niche users.


$ \Rightarrow $ The results show that neural network models will have good material to exploit.

The PrefixScan model can be enabled in the script to provide images to explore sequences
for actions or as precursor to Sequential DNN models.


## 4. Define Data and Concept Drifts
After exploring the data inputs to the model, we define monitoring for data and concept shifts.

let X = features

let Y = targets

Data shift is a change in the joint distribution, P(X, Y). 

Using the probability product rule, we can explore 4 causes for the
simplest changes in P(X,Y)
$$ P(X, Y) = P(Y, X) = P(X|Y)P(Y) = P(Y|X)P(X) $$

We can look for changes in one member in the following pairs at any time
(one is simpler than exploring more than 1 member changing at same time):
$$ P(X|Y) * P(Y) $$
$$ P(Y|X) * P(X) $$

* Covariate shift:
  $$ P(X) changed,  P(Y|X) unchanged $$
  Distr of model inputs changes.
* Label shift:
  $$ P(Y) changed,  P(X|Y) unchanged $$
  Distr of model outputs changes, but for any given output, the input distribution stays the same.
  
* Concept shift:
  $$ P(X) unchanged,  P(Y|X) changed $$
* Manifestation shift:
  $$ P(Y) unchanged,  P(X|Y) changed $$

[see more at NannyML](https://www.nannyml.com/blog/concept-drift)

if the train, eval, and test split are random, then one could use
the maximum differences in their distributions as a lower limit on 
the estimate for stochastic error.  A trigger for data drift should
then be GEQ about 3 times that stochastic error.

TODO: continue with this... have already partitioned the ratings by timestamp into 2 sets...

### Data [Drift and Skew using TFX ExampleValidator component](https://www.tensorflow.org/tfx/guide/exampleval)

[see also TFDV](https://www.tensorflow.org/tfx/tutorials/data_validation/tfdv_basic)
* Drift
  Drift detection is supported for categorical features and between consecutive spans of data
  (i.e., between span N and span N+1), such as between different days of training data.
    * L-infinity distance
    * alerts when drift is higher than threshold distance
* Skew
    * data - schema skew
      occurs when the training and serving data do not conform to the same schema.
      Any expected deviations between the two (such as the label feature being only present in the training data but not in serving) should be specified through environments field in the schema.
    * feature skew
      occurs when the training and serving feature values are different
      This can happen when:
        * A data source of features is modified between training and serving time
        * A difference in logic for generating between training and serving.
    * distribution skew
      occurs when the training and serving distributions are significantly different.
      some key causes:
        * different code or different data sources to generate the training dataset
        * a faulty sampling mechanism that chooses a non-representative subsample of the serving data to train on.



%%bash
## This is a GPU cell, showing configuration needed to run on T4x2 GPUs.
### It has been toggled to a MarkDown to not run it.
date

. /usr/local/bin/activate
conda activate my_tfx_env

nvidia-smi

cd /kaggle/working/recommender_systems

##### Point linker to the pip-installed CUDA libraries
NVIDIA_DIR=$(python3 -c "import site; print(site.getsitepackages()[0] + '/nvidia')")
##### Find all 'lib' directories inside the nvidia folder and append to LD_LIBRARY_PATH
export LD_LIBRARY_PATH=$LD_LIBRARY_PATH:$(find $NVIDIA_DIR -name "lib" -type d | tr '\n' ':')
##### ---------------------------------------------------------

#for running on GPUs:
export KERAS_BACKEND="tensorflow"
export TF_FORCE_GPU_ALLOW_GROWTH="true"
export CUDA_VISIBLE_DEVICES="0,1"
export TF_CPP_MIN_LOG_LEVEL="0"
export PYTHONUNBUFFERED=1
export NCCL_P2P_DISABLE=1
export NCCL_IB_DISABLE=1
export NCCL_SOCKET_IFNAME="lo"
export XLA_PYTHON_CLIENT_PREALLOCATE="false"
export XLA_PYTHON_CLIENT_ALLOCATOR="platform"

echo "--- DIRECT TENSORFLOW GPU CHECK ---"
python3 -c "import tensorflow as tf; print('\nBUILT WITH CUDA:', tf.test.is_built_with_cuda()); print('PHYSICAL GPUS:', tf.config.list_physical_devices('GPU'))"

python3 src/test/python/movie_lens_tfx/gpus_avail_test.py

date

## 5. Run the data pre-processing pipeline and look at results.
the Transformer component performs operations on the raw data.
The operations are defined in the preprocessing_fn in file [transform_movie_lens.py](https://github.com/nking/recommender_systems/blob/9820c1b21a05a04497cff50d69ee9df7ed1b1f3f/src/main/python/movie_lens_tfx/transform_movie_lens.py#L105)

## 6. Run the Baseline and Production model pipelines and look at results.


In [ ]:
%%bash
date

. /usr/local/bin/activate
conda activate my_tfx_env

cd /kaggle/working/recommender_systems

#for running on GPUs:
export KERAS_BACKEND="tensorflow"
export PYTHONUNBUFFERED=1
export TF_CPP_MIN_LOG_LEVEL=2  # 0=all, 1=INFO suppressed, 2=INFO+WARNING suppressed, 3=ERROR only

python src/test/python/movie_lens_tfx/run_kaggle_pipelines.py > pipeline_logs.txt 2>&1

date

# Allow the IOPub channel to drain before the cell finishes
sleep 5

## 7. Explore the model evaluation
The best model is found by search with the TFX Tuner component.
The subsequent trained model can be compared against a baseline model in order to examine whether the model metrics are improved enough and whether business value metrics are met. [Evaluation](https://github.com/nking/recommender_systems/blob/9820c1b21a05a04497cff50d69ee9df7ed1b1f3f/src/main/python/movie_lens_tfx/PipelineComponentsFactory.py#L287)

In [ ]:
%%bash
. /usr/local/bin/activate
conda activate my_tfx_env

cd /kaggle/working/recommender_systems
export PIPELINE_DATA_DIR=/kaggle/working/recommender_systems/bin/rs_pipeline
export OUTPUT_BASE_DIR=/kaggle/working
export TAG="latest"
python3 -m unittest src.test.python.movie_lens_tfx.write_tensorboard_pngs_test.WriteTensorboardToPng
#writes to /kaggle/working/pngs

### Model training and validation losses and metrics
Train, validation learning curves:

In [ ]:
from pathlib import Path
from IPython.display import Image, Markdown, display

png_dir = Path("/kaggle/working/pngs")

for png_path in sorted(png_dir.glob("*.png")):
    display(Markdown(f"#### `{png_path.name}`"))
    display(Image(filename=str(png_path)))

### Model training and validation losses and metrics (from repository)



<img src="https://raw.githubusercontent.com/nking/recommender_systems/refs/heads/main/docs/mlops/best_fitting_model_stats/twotowerdnn_softmax/snapshots_tensorboard/epoch_composite_ndcg_100.png">

<img src="https://raw.githubusercontent.com/nking/recommender_systems/refs/heads/main/docs/mlops/best_fitting_model_stats/twotowerdnn_softmax/snapshots_tensorboard/epoch_ndcg_head_100.png">

<img src="https://raw.githubusercontent.com/nking/recommender_systems/refs/heads/main/docs/mlops/best_fitting_model_stats/twotowerdnn_softmax/snapshots_tensorboard/epoch_ndcg_torso_100.png">

<img src="https://raw.githubusercontent.com/nking/recommender_systems/refs/heads/main/docs/mlops/best_fitting_model_stats/twotowerdnn_softmax/snapshots_tensorboard/epoch_ndcg_tail_100.png">

<img src="https://raw.githubusercontent.com/nking/recommender_systems/refs/heads/main/docs/mlops/best_fitting_model_stats/twotowerdnn_softmax/snapshots_tensorboard/epoch_mean_loss.png">

<img src="https://raw.githubusercontent.com/nking/recommender_systems/refs/heads/main/docs/mlops/best_fitting_model_stats/twotowerdnn_softmax/snapshots_tensorboard/epoch_recall_100.png">

<img src="https://raw.githubusercontent.com/nking/recommender_systems/refs/heads/main/docs/mlops/best_fitting_model_stats/twotowerdnn_softmax/snapshots_tensorboard/epoch_hit_rate.png">

<img src="https://raw.githubusercontent.com/nking/recommender_systems/refs/heads/main/docs/mlops/best_fitting_model_stats/twotowerdnn_softmax/snapshots_tensorboard/epoch_precision_100.png">

<img src="https://raw.githubusercontent.com/nking/recommender_systems/refs/heads/main/docs/mlops/best_fitting_model_stats/twotowerdnn_softmax/snapshots_tensorboard/epoch_logit_max.png">

<img src="https://raw.githubusercontent.com/nking/recommender_systems/refs/heads/main/docs/mlops/best_fitting_model_stats/twotowerdnn_softmax/snapshots_tensorboard/epoch_logit_mean.png">

<img src="https://raw.githubusercontent.com/nking/recommender_systems/refs/heads/main/docs/mlops/best_fitting_model_stats/twotowerdnn_softmax/snapshots_tensorboard/epoch_logit_min.png">

### Movie Embedding UMAP plots
<img src="https://raw.githubusercontent.com/nking/recommender_systems/refs/heads/main/docs/mlops/best_fitting_model_stats/twotowerdnn_softmax/post_training_analysis/embedding_plots/tower_embed_umap_users_user_tier.png">

<img src="https://raw.githubusercontent.com/nking/recommender_systems/refs/heads/main/docs/mlops/best_fitting_model_stats/twotowerdnn_softmax/post_training_analysis/embedding_plots/tower_embed_umap_movies_movie_tier.png">

<img src="https://raw.githubusercontent.com/nking/recommender_systems/refs/heads/main/docs/mlops/best_fitting_model_stats/twotowerdnn_softmax/post_training_analysis/embedding_plots/tower_embed_umap_joint_users_movies.png">

<img src="https://raw.githubusercontent.com/nking/recommender_systems/refs/heads/main/docs/mlops/best_fitting_model_stats/twotowerdnn_softmax/post_training_analysis/embedding_plots/tower_embed_umap_catalog_anchored_joint_users_movies.png">



## 8. Model analysis
The training of the TwoTowerDNN used an inbatch softmax loss with log q-corrections, and used the metric composite_ndcg@100 on the validation set for early stopping
and model selection by the keras tuner BayesianOptimization.

composite_ndcg@100 is a compostion of ndcg@100 for each of the head, torso and tail of the ratings frequency distribution.

The model analysis can be run from [post_training.sh](https://raw.githubusercontent.com/nking/recommender_systems/refs/heads/main/post_training.sh)
which uses a [retrieval project script](https://raw.githubusercontent.com/nking/retrieval/refs/heads/main/src/test/python/movie_lens_retrieval/post_training_analysis.py)

Example of the last saved model analysis is below showing conclusions then the detailed numbers.   Also shown are the movie embeddings plotted with t-SNE and UMAP.

In [ ]:
import requests
from rich import print_json

url = "https://raw.githubusercontent.com/nking/recommender_systems/refs/heads/main/docs/mlops/best_fitting_model_stats/twotowerdnn_softmax/post_training_analysis/summary_conclusions.json"
response = requests.get(url)

# Prints syntax-highlighted, nicely indented JSON
print_json(response.text)

url2 = "https://raw.githubusercontent.com/nking/recommender_systems/refs/heads/main/docs/mlops/best_fitting_model_stats/twotowerdnn_softmax/post_training_analysis/summary_metrics.json"
response = requests.get(url2)

# Prints syntax-highlighted, nicely indented JSON
print_json(response.text)

## 9. Link to business value

TODO: list supplemental data that could be used to link to business value:
watch time, click through rate, conversion rate, user engagement, churn, revenue uplift from associated ads.

TODO: quantifying the business value:
A/B testing with control group and test group where the later receives ranked recommendation from the new model.  These notes are in the ranker project.

## 10. Monitor the model

The model deployed to production can be monitored to detect when the business utility drops below an acceptable threshold.
The business utility here is linked to the accuracy of recommending movies to the user that they would like and haven't seen.

Detecting model degradation when users have watched a movie, and before they have rated them, requires bootstrapping
using an extra model trained from reference data that has the user ratings. 

NannyML is a popular library used for detecting changes in ML model features, targets, and relationships between
them.  
The [NannyML Model performance estimator](https://nannyml.readthedocs.io/en/stable/how_it_works/performance_estimation.html#dle-implementation-details) for regression  is their Direct Loss Estimator algorithm (DLE).

The models:
* f = the monitored recommender model
* h = the Nanny model (discussed below)

The datasets used:
* train: used to train f.  contains features X and targets Y where targets are the ratings.
* reference: used to train h. contains features X and targets Y
* analysis: recent data.  contains only features X
The datasets are ordered by time.

The nanny model h is an extra model used for degradation monitoring.
* h is trained with X_reference, and y_reference
    * for each observation in X, calculate absolute error AE_reference = | f(X_reference) - y_reference |
    * train A_reference = h(X_reference, f(X_reference))
          * DLE for regression uses LightGBM
* predict AE_analysis using h
    * for each observation in X: AE_analysis_est = h( X_analysis, f(X_analysis))
    * MAE_analysis_est = mean of the AE_analysis_est
* the variance of large chunks of data are determined and the upper and lower bounds for anomalies are set using the sample error using those variances.  When a predicted MAE exceed the thresholds, an alert can be triggered and root cause analysis can be performed to decide if the model should be retrained or refactored and retrained as part of a Continuous Training (CT) workflow.

DLE assumes:
* no concept drift
* no covariate shift to previously unseen regions of input space
* noise is heteroscedastic around the monitored model target, y,
  and it is dependent on the monitored model input features
* the dataset to be analyzed is large enough so that the performance estimate is not dominated by sampling error

Note: [AI Stories Podcasts](https://www.youtube.com/@aistoriespodcast) has great interviews helpful in AI/ML, such as [Advanced Monitoring using NannyML with Wojtek Kuberski](https://youtu.be/PggImbPX7kg?si=41ThyCHKwxkoI5mU) and [Polars: Fast, Efficient Data Manipulation with Ritchie Vink](https://youtu.be/MakwbONEW1Y?si=7RKh9M9hbGteCB0F)

#### Model monitoring is a separate process from the serving inference, so an application can be built for it or cloud services can be used.

TODO: add cloud model monitoring options and details...
Apache Airflow,  Vertex AI Model Monitoring, Azure ML Model monitoring, Amazon SageMaker Model monitor, NannyML is an option through the marketplaces in Azure and Amazon SageMaker, ...

## Summary
TFX pipelines are valuable tools in MLOps
* highly scalable
    * distributed data pre-processing
    * distributed model training
* configurable to run on a variety of architectures
    * e.g. TPUs, GPUs for accelerated training
* configurable for variety of different data source types
* configurable for fault tolerance and failover
* configurable for different types of pipeline storage and model registry
* configurable for cloud native, cloud specific, and local environments
* robust, consistent processing
* configurable exported model types compatible with keras, pytorch, jax:
    * default saved_models
    * liteRT for embedded and IOT environments
    * TensorflowJS for browsers
* usable in cloud environments
    * compatible with many orchestrators
        * Apache Airflow, Apache Beam, Kubeflow pipelines
    * can be used in containers (required by kubernetes for example)
    * compatible with cluster management systems like kubernetes, flink, spark, and cloud native offerings
    * can be compiled into Kubeflow pipelines
    * can be run from a command line interface in the cloud or locally
* the MLMD store is usable by a variety of monitoring systems including cloud-specific systems
* Tensorboard for visualization of model training output, TFDV for visualization of data
* compatible with use of BI tools like GCP's Looker, DataPrep
* inter-operative with Vertex AI Training, Prediction, Dataflow

## What's next

The ranker tier is in the Kaggle notebook [Ranker cross-encoder w/ GATv2](https://www.kaggle.com/code/nicholeasuniquename/ranker-cross-encoder-w-gatv2)
